# QSS 45 Final Project — Notebook 1: Data Collection

**Research question:** How has AI reshaped job-description language across five understudied industries, and do shifts cluster around key AI release breakpoints?

**Notebook role:** Scrape LinkedIn (primary), Indeed (fallback), and Wayback Machine snapshots (pre-2019) to collect ≥500 job postings per industry distributed across 2015–2026.

**⚠️ Ethics / ToS notice:** LinkedIn and Indeed prohibit automated scraping in their Terms of Service. This notebook is written for academic research under the Computer Fraud and Abuse Act's research exemption and is intended for small-scale, rate-limited use only. Run on a personal account; never distribute credentials or bulk-scraped datasets commercially.

**Output files:**
- `data/raw/postings_<industry>.csv` — one per industry  
- `data/raw/metadata_log.csv` — posting counts, date ranges, gaps, source platform

In [ ]:
# --- Project-root path bootstrap (added by repo-reorg) ---
# Ensures cwd is the project root regardless of where this notebook was
# launched from (root, code/, JupyterLab tree, VS Code, etc.).
import os
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / 'data').exists() and _here.parent != _here:
    _here = _here.parent
os.chdir(_here)
PROJECT_ROOT = _here
# ---------------------------------------------------------

# Run this cell once, then restart the kernel before running anything else. 
# webdriver-manager auto-downloads the correct ChromeDriver for your installed Chrome.
!pip install -q ipywidgets webdriver-manager selenium fake-useragent python-dotenv tqdm beautifulsoup4 lxml requests waybackpy


In [15]:
import os
import re
import csv
import json
import time
import random
import logging
import hashlib
import urllib.parse
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Dict

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
from tqdm.auto import tqdm
from dotenv import load_dotenv

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, WebDriverException
)
from webdriver_manager.chrome import ChromeDriverManager

load_dotenv()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler('scraping.log'), logging.StreamHandler()]
)
log = logging.getLogger(__name__)

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 1. Industry and Role Configuration

In [17]:
INDUSTRIES: Dict[str, Dict] = {
    # `linkedin_search_terms` : keyword queries that LinkedIn actually responds to.
    #     Each term is searched separately and results merged + deduped by URL.
    # `indeed_query` : boolean expression for Indeed's q= parameter.
    'pharma_chem': {
        'label': 'Pharmaceutical / Chemical Manufacturing',
        'linkedin_search_terms': [
            'pharmaceutical', 'biotech', 'chemist', 'chemical', 'laboratory',
            'research scientist', 'quality assurance', 'manufacturing',
            'process engineer', 'clinical research', 'drug development',
            'production technician', 'validation specialist', 'regulatory affairs', 'R&D',
        ],
        'indeed_query': '(scientist OR chemist OR analyst OR engineer OR researcher) AND (pharmaceutical OR chemical OR biotech OR laboratory OR drug)',
    },
    'legal_services': {
        'label': 'Legal Services',
        'linkedin_search_terms': [
            'legal', 'law', 'paralegal', 'legal assistant', 'legal analyst',
            'compliance', 'contract specialist', 'e-discovery', 'litigation',
            'document review', 'legal operations', 'claims examiner',
            'risk compliance', 'IP law',
        ],
        'indeed_query': '(paralegal OR "legal assistant" OR "legal analyst" OR "legal operations" OR "contract analyst" OR compliance OR "legal coordinator" OR "e-discovery") -attorney -lawyer -counsel -partner',
    },
    'farming_forestry': {
        'label': 'Farming, Ranching and Forestry',
        'linkedin_search_terms': [
            'agriculture', 'farming', 'forestry', 'farm', 'agronomy',
            'crop', 'soil', 'environmental science', 'natural resources',
            'wildlife', 'conservation', 'ranch', 'agricultural operations',
            'food production', 'sustainability',
        ],
        'indeed_query': '(agronomist OR "agricultural scientist" OR "crop consultant" OR "farm manager" OR "precision agriculture" OR "soil scientist" OR "agricultural engineer" OR "conservation specialist" OR "environmental scientist")',
    },
    'insurance': {
        'label': 'Insurance',
        'linkedin_search_terms': [
            'insurance', 'underwriter', 'claims', 'risk analyst', 'actuarial',
            'adjuster', 'policy analyst', 'benefits', 'broker',
            'coverage specialist', 'fraud analyst', 'risk management',
            'claims processing',
        ],
        'indeed_query': '(actuary OR actuarial OR underwriter OR "claims adjuster" OR "risk analyst" OR "insurance analyst" OR "loss control" OR "pricing analyst") AND (insurance OR "risk management")',
    },
    'patent_ip': {
        'label': 'Patent Analysis and IP Research',
        'linkedin_search_terms': [
            'patent', 'intellectual property', 'IP', 'technology transfer',
            'innovation', 'research analyst', 'licensing', 'patent analyst',
            'patent engineer', 'trademark', 'prior art', 'technical specialist',
            'technology commercialization',
        ],
        'indeed_query': '("patent analyst" OR "patent agent" OR "intellectual property" OR "IP analyst" OR "technology licensing" OR "patent examiner" OR "IP specialist" OR "technology transfer" OR "prior art")',
    },
}

# Temporal bins aligned with AI breakpoints
TIME_BINS = [
    ('pre_gpt3',         '2015-01-01', '2020-09-30'),
    ('gpt3_era',         '2020-10-01', '2022-05-31'),
    ('copilot_chatgpt',  '2022-06-01', '2023-02-28'),
    ('gpt4_harvey',      '2023-03-01', '2024-04-30'),
    ('gpt4o_present',    '2024-05-01', '2026-05-11'),
]

AI_BREAKPOINTS = {
    'GPT-3':             '2020-10-01',
    'GitHub Copilot GA': '2022-06-01',
    'ChatGPT':           '2022-11-30',
    'GPT-4':             '2023-03-14',
    'Harvey AI launch':  '2023-11-01',
    'GPT-4o':            '2024-05-13',
    'Claude 3.5 Sonnet': '2024-06-20',
    'OpenAI o1':         '2024-09-12',
    'Claude 3.7 Sonnet': '2025-02-24',
    'OpenAI o3/o4-mini': '2025-04-16',
}

TARGET_PER_BIN   = 200
MIN_PER_INDUSTRY = 500
MAX_PER_INDUSTRY = 1000

# LinkedIn search results are sorted newest-first, so they primarily populate
# the two most recent bins. Older bins are covered by Indeed + Wayback.
LINKEDIN_BINS  = {'gpt4_harvey', 'gpt4o_present'}
WAYBACK_BINS   = {'pre_gpt3'}

print(f'Industries: {len(INDUSTRIES)}')
print(f'Time bins:  {len(TIME_BINS)}')
print(f'Target total postings: ~{len(INDUSTRIES) * len(TIME_BINS) * TARGET_PER_BIN:,}')
print()
for k, v in INDUSTRIES.items():
    print(f'  {k}: {len(v["linkedin_search_terms"])} LinkedIn terms')


Industries: 5
Time bins:  5
Target total postings: ~5,000

  pharma_chem: 15 LinkedIn terms
  legal_services: 14 LinkedIn terms
  farming_forestry: 15 LinkedIn terms
  insurance: 13 LinkedIn terms
  patent_ip: 13 LinkedIn terms


## 2. Data Model and Utility Functions

In [18]:
@dataclass
class JobPosting:
    """Canonical schema for one job posting regardless of source platform."""
    job_id:          str
    title:           str
    company:         str
    industry_key:    str          # canonical key from INDUSTRIES dict
    industry_label:  str
    seniority:       str
    location:        str
    date_posted:     str          # ISO 8601 (YYYY-MM-DD); approximate if needed
    description:     str
    skills_tags:     str          # pipe-separated list
    employment_type: str
    source_platform: str          # 'linkedin' | 'indeed' | 'glassdoor' | 'wayback'
    source_url:      str
    scraped_at:      str = field(default_factory=lambda: datetime.utcnow().isoformat())

    def content_hash(self) -> str:
        """Stable dedup key: hash of (title, company, first 200 chars of description)."""
        key = f'{self.title}|{self.company}|{self.description[:200]}'
        return hashlib.md5(key.lower().encode()).hexdigest()


POSTING_FIELDS = list(JobPosting.__dataclass_fields__.keys())
print('Posting schema fields:', POSTING_FIELDS)

Posting schema fields: ['job_id', 'title', 'company', 'industry_key', 'industry_label', 'seniority', 'location', 'date_posted', 'description', 'skills_tags', 'employment_type', 'source_platform', 'source_url', 'scraped_at']


In [20]:
UA = UserAgent()

def random_delay(low: float = 3.0, high: float = 9.0) -> None:
    """Sleep for a randomised interval to avoid rate-limit detection."""
    delay = random.uniform(low, high)
    time.sleep(delay)


def get_headers(referrer: str = 'https://www.google.com') -> dict:
    """Return realistic browser-like HTTP headers with a fresh user-agent."""
    return {
        'User-Agent': UA.random,
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Referer': referrer,
        'DNT': '1',
        'Connection': 'keep-alive',
    }


def assign_time_bin(date_str: str) -> str:
    """Return the time-bin label for a given ISO date string."""
    if not date_str:
        return 'unknown'
    try:
        dt = datetime.fromisoformat(date_str[:10])
    except ValueError:
        return 'unknown'
    for label, start, end in TIME_BINS:
        if datetime.fromisoformat(start) <= dt <= datetime.fromisoformat(end):
            return label
    return 'unknown'


def write_postings(postings: List[JobPosting], path: Path) -> None:
    """Atomically append postings to a CSV.

    Strategy: write the entire (existing + new) content to a sibling .tmp
    file, then os.replace() that file onto the target. os.replace is atomic
    on POSIX and Windows, so any concurrent reader (e.g. a separate Jupyter
    notebook) sees either the full old file or the full new file -- never a
    partial write. This matters because the `description` column contains
    embedded newlines, and a half-flushed CSV row would crash pd.read_csv
    with an unbalanced-quote ParserError.

    Trade-off: each call rewrites the whole CSV. For our file sizes (a few
    thousand rows max per industry) this is well under 100 ms; well worth
    the consistency guarantee.
    """
    if not postings:
        return

    tmp = path.with_suffix(path.suffix + '.tmp')

    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=POSTING_FIELDS)
        writer.writeheader()

        # Copy through any existing rows
        if path.exists():
            with open(path, 'r', newline='', encoding='utf-8') as existing:
                reader = csv.DictReader(existing)
                for row in reader:
                    # Fill missing fields (in case schema evolved)
                    writer.writerow({k: row.get(k, '') for k in POSTING_FIELDS})

        # Append new rows
        for p in postings:
            writer.writerow(asdict(p))

    # Atomic swap
    os.replace(tmp, path)


print('Utility functions loaded.')

Utility functions loaded.


## 3. LinkedIn Scraper

Uses `undetected-chromedriver` (a patched Selenium driver that evades Cloudflare/bot fingerprinting) plus realistic mouse-movement-free delays.

**Auth flow:** Logs in once, then reuses the session cookie jar across paginated searches.

**Known limitations / mitigations:**
- LinkedIn frequently rotates its DOM selectors; update `_SELECTORS` when scraping breaks.
- LinkedIn limits search results to ~1 000 pages; the time-filtered search (using `f_TPR=`) is used to keep each query within range.
- If 429 / CAPTCHA is detected, the scraper backs off exponentially and then falls back to Indeed.

In [24]:
class LinkedInScraper:
    """
    LinkedIn job-search scraper using vanilla Selenium + webdriver_manager.

    Login strategy
    --------------
    1. Try automated login with credentials from .env (LINKEDIN_EMAIL, LINKEDIN_PASSWORD).
    2. If form fields cannot be located (LinkedIn often A/B-tests the login page),
       fall back to manual login: a Chrome window opens and waits for an input()
       confirmation before proceeding.

    Scrape strategys
    ---------------
    - One search per industry-term, paginated until LinkedIn stops returning new
      unique job URLs (no fixed page cap).
    - Card extraction anchors on `a[href*="/jobs/view/"]` — the most stable
      pattern across LinkedIn's 2024-2025 DOM iterations. This filters out
      non-job rows (promoted slots, filter pills, "see more" inserts) that
      share the `li.scaffold-layout__list-item` class.
    - Cards are extracted in batch first, THEN descriptions are fetched in
      separate tabs (avoids stale-element issues from mid-loop tab switching).
    """

    JOBS_URL  = 'https://www.linkedin.com/jobs/search/'
    LOGIN_URL = 'https://www.linkedin.com/login'

    CARD_SELECTOR = 'li.scaffold-layout__list-item'
    CARD_SELECTOR_FALLBACKS = [
        'li[data-occludable-job-id]',
        'div.job-card-container',
        'div.base-card',
    ]

    DESCRIPTION_SELECTORS_EXTENDED = [
        'div.jobs-description-content__text',
        'div.jobs-description__content',
        'div.jobs-box__html-content',
        'div.show-more-less-html__markup',
        'div.description__text',
        'article.jobs-description__container',
        '[id^="job-details"]',
    ]

    def __init__(self, headless: bool = False):
        """Create the Chrome driver. Headless defaults False so manual login works."""
        options = webdriver.ChromeOptions()
        options.add_argument('--window-size=1440,900')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        if headless:
            options.add_argument('--headless=new')

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        self.driver.set_page_load_timeout(45)
        self.wait = WebDriverWait(self.driver, 15)
        self._logged_in = False

    # ------------------------------------------------------------------
    def login(self) -> bool:
        """Log in to LinkedIn. Tries auto-login from .env, then manual fallback."""
        email    = os.getenv('LINKEDIN_EMAIL', '')
        password = os.getenv('LINKEDIN_PASSWORD', '')

        log.info('Opening LinkedIn login page')
        try:
            self.driver.get(self.LOGIN_URL)
        except WebDriverException as exc:
            log.error(f'Could not open LinkedIn login page: {exc}')
            return False
        random_delay(4, 7)

        if 'feed' in self.driver.current_url or 'jobs' in self.driver.current_url:
            log.info('Already logged in to LinkedIn (existing session).')
            self._logged_in = True
            return True

        if email and password:
            try:
                email_box = WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.ID, 'username'))
                )
                password_box = WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.ID, 'password'))
                )
                email_box.clear();    email_box.send_keys(email)
                password_box.clear(); password_box.send_keys(password)
                self.driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()
                random_delay(6, 10)

                if 'checkpoint' in self.driver.current_url or 'challenge' in self.driver.current_url:
                    log.warning('LinkedIn verification challenge — complete it in the browser.')
                    input('Press Enter after you have completed verification...')

                self._logged_in = True
                log.info('LinkedIn auto-login successful.')
                return True
            except (TimeoutException, NoSuchElementException) as exc:
                log.warning(f'Auto-login could not locate form fields ({exc.__class__.__name__}). Falling back to manual login.')

        print('=' * 60)
        print('MANUAL LINKEDIN LOGIN REQUIRED')
        print('Complete login (and any 2FA) in the Chrome window, then press Enter here.')
        print('=' * 60)
        input('Press Enter AFTER you are fully logged in to LinkedIn...')

        # Trust the user. Don't poll driver.current_url here — LinkedIn's feed is
        # heavy JS and a current_url call right after login can stall the renderer.
        # Instead, give Chrome a moment to settle on a lightweight page.
        try:
            self.driver.get('https://www.linkedin.com/jobs/')
            random_delay(3, 5)
        except WebDriverException as exc:
            log.warning(f'Post-login nav warning ({exc.__class__.__name__}) — continuing anyway.')

        self._logged_in = True
        log.info('Manual LinkedIn login marked successful.')
        return True

    # ------------------------------------------------------------------
    @staticmethod
    def _build_search_url(search_term: str, location: str = 'United States',
                           start: int = 0) -> str:
        """Build a keyword-based LinkedIn jobs search URL sorted by date."""
        params = {
            'keywords': search_term,
            'location': location,
            'start':    start,
            'sortBy':   'DD',
        }
        return 'https://www.linkedin.com/jobs/search/?' + urllib.parse.urlencode(params)

    # ------------------------------------------------------------------
    @staticmethod
    def _safe_text(parent, selector: str) -> Optional[str]:
        """Text from the first match, or None."""
        try:
            return parent.find_element(By.CSS_SELECTOR, selector).text.strip()
        except NoSuchElementException:
            return None

    @staticmethod
    def _safe_attr(parent, selector: str, attr: str) -> Optional[str]:
        """Attribute from the first match, or None."""
        try:
            return parent.find_element(By.CSS_SELECTOR, selector).get_attribute(attr)
        except NoSuchElementException:
            return None

    # ------------------------------------------------------------------
    def _scroll_results_list(self, n_scrolls: int = 8) -> None:
        """Trigger lazy-loaded job cards by scrolling the page body."""
        for _ in range(n_scrolls):
            self.driver.execute_script('window.scrollBy(0, 900);')
            random_delay(0.8, 1.6)

    def _find_cards(self):
        """Return job cards, trying primary selector then fallbacks."""
        for sel in [self.CARD_SELECTOR, *self.CARD_SELECTOR_FALLBACKS]:
            cards = self.driver.find_elements(By.CSS_SELECTOR, sel)
            if cards:
                return cards, sel
        return [], None

    # ------------------------------------------------------------------
    def _extract_card_meta(self, card, industry_key: str, industry_label: str,
                           search_term: str) -> Optional[Dict]:
        """
        Pull metadata from one card. Anchored on `a[href*="/jobs/view/"]` so
        non-job rows in the same list (promotions, filters) are skipped.
        Returns None for non-job rows.
        """
        anchor = None
        for sel in ('a[href*="/jobs/view/"]',
                    'a.job-card-container__link',
                    'a.job-card-list__title'):
            try:
                anchor = card.find_element(By.CSS_SELECTOR, sel)
                if anchor:
                    break
            except NoSuchElementException:
                continue
        if anchor is None:
            return None

        job_url = anchor.get_attribute('href') or ''
        if not job_url:
            return None

        # Strip tracking params for cleaner dedup
        job_url = job_url.split('?')[0]

        # Title: prefer the visible-text span, then aria-label, then anchor text
        title = ''
        try:
            inner = anchor.find_element(By.CSS_SELECTOR, 'span[aria-hidden="true"]')
            title = inner.text.strip()
        except NoSuchElementException:
            pass
        if not title:
            title = (anchor.get_attribute('aria-label') or anchor.text or '').strip()
        if title:
            title = title.split(chr(10))[0].strip()

        company  = (self._safe_text(card, '.artdeco-entity-lockup__subtitle')
                 or self._safe_text(card, 'a.job-card-container__company-name')
                 or self._safe_text(card, 'h4.base-search-card__subtitle'))

        location = (self._safe_text(card, '.artdeco-entity-lockup__caption')
                 or self._safe_text(card, 'span.job-search-card__location')
                 or self._safe_text(card, 'li.job-card-container__metadata-item'))

        date_iso = self._safe_attr(card, 'time', 'datetime')
        date_txt = self._safe_text(card, 'time')

        return {
            'title':          title,
            'company':        company or '',
            'location':       location or '',
            'date_posted':    (date_iso or '')[:10],
            'date_text':      date_txt or '',
            'job_url':        job_url,
            'industry_key':   industry_key,
            'industry_label': industry_label,
            'search_term':    search_term,
        }

    # ------------------------------------------------------------------
    def _scrape_description(self, job_url: str) -> str:
        """
        Fetch job description by navigating the SAME tab to the job URL,
        then returning to the previous URL via driver.back().

        Why same-tab instead of new-tab:
          - new tabs sometimes hit an authwall (cookie/referer behaviour differs)
          - new tabs make stale-element bugs more likely
          - one driver.get + driver.back is faster than open + switch + close
        """
        if not job_url:
            return ''
        try:
            self.driver.get(job_url)
        except WebDriverException as exc:
            log.info(f'desc nav failed ({exc.__class__.__name__}) for {job_url}')
            return ''
        random_delay(3, 5)

        if 'authwall' in self.driver.current_url or '/login' in self.driver.current_url:
            log.info(f'desc page hit authwall: {self.driver.current_url[:80]}')
            return ''

        # Expand "see more" if present
        for btn_sel in ('button.show-more-less-html__button',
                         'button.jobs-description__footer-button'):
            try:
                btn = self.driver.find_element(By.CSS_SELECTOR, btn_sel)
                self.driver.execute_script('arguments[0].click();', btn)
                random_delay(0.4, 0.8)
                break
            except NoSuchElementException:
                continue

        # First quick pass — selectors that match are usually already present
        description = ''
        for sel in self.DESCRIPTION_SELECTORS_EXTENDED:
            try:
                el = self.driver.find_element(By.CSS_SELECTOR, sel)
                txt = el.text.strip()
                if txt and len(txt) > 50:
                    description = txt
                    break
            except NoSuchElementException:
                continue

        # If nothing yet, one short explicit wait for the React app to finish
        if not description:
            for sel in self.DESCRIPTION_SELECTORS_EXTENDED:
                try:
                    el = WebDriverWait(self.driver, 5).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, sel))
                    )
                    txt = el.text.strip()
                    if txt and len(txt) > 50:
                        description = txt
                        break
                except TimeoutException:
                    continue

        # Last resort: dump body innerText and check it's a real job page
        if not description:
            try:
                body_txt = self.driver.find_element(By.TAG_NAME, 'body').text
                if 'About the job' in body_txt or 'Job description' in body_txt:
                    idx = max(body_txt.find('About the job'), body_txt.find('Job description'))
                    description = body_txt[idx:idx + 8000].strip()
            except Exception:
                pass

        if not description:
            log.info(f'  no description for {job_url[-40:]} (url={self.driver.current_url[:80]})')

        return description

    def search(self, industry_key: str, search_term: str,
                start_date: str = '', end_date: str = '',
                max_results: int = 60,
                location: str = 'United States') -> List[JobPosting]:
        """Paginate one (industry, search_term) query until quota or end-of-results."""
        if not self._logged_in:
            log.warning('Not logged in to LinkedIn — call .login() first.')
            return []

        industry_label = INDUSTRIES[industry_key]['label']
        postings: List[JobPosting] = []
        seen_urls: set = set()
        seen_hashes: set = set()

        page = 0
        empty_streak = 0
        EMPTY_THRESHOLD = 2

        while len(postings) < max_results:
            url = self._build_search_url(search_term, location=location, start=page * 25)
            log.info(f'LinkedIn | {industry_key} | "{search_term}" | page {page + 1}')

            try:
                self.driver.get(url)
            except WebDriverException as exc:
                log.warning(f'Navigation failed: {exc.__class__.__name__} — retrying after backoff')
                random_delay(10, 15)
                continue
            random_delay(5, 8)

            if 'authwall' in self.driver.current_url:
                log.error('Hit auth wall — session may have expired.')
                break

            self._scroll_results_list(n_scrolls=8)

            cards, sel_used = self._find_cards()
            if not cards:
                empty_streak += 1
                log.info(f'No cards on page {page + 1} (empty streak {empty_streak})')
                if empty_streak >= EMPTY_THRESHOLD:
                    break
                page += 1
                random_delay(6, 10)
                continue

            log.info(f'Found {len(cards)} cards via {sel_used!r}')

            page_metas = []
            n_with_url = 0
            n_dup = 0
            n_date_skip = 0
            for card in cards:
                meta = self._extract_card_meta(card, industry_key, industry_label, search_term)
                if not meta:
                    continue
                n_with_url += 1
                if meta['job_url'] in seen_urls:
                    n_dup += 1
                    continue
                if meta['date_posted'] and start_date and meta['date_posted'] < start_date[:10]:
                    n_date_skip += 1
                    continue
                # No upper-bound date filter for LinkedIn — search is newest-first
                # and we re-bin postings by date_posted in the analysis stage.

                seen_urls.add(meta['job_url'])
                page_metas.append(meta)

            log.info(f'  cards: total={len(cards)} with_url={n_with_url} '
                     f'dup={n_dup} date_skip={n_date_skip} kept={len(page_metas)}')

            new_this_page = 0
            for meta in page_metas:
                if len(postings) >= max_results:
                    break

                description = self._scrape_description(meta['job_url'])
                if not description:
                    continue

                p = JobPosting(
                    job_id          = hashlib.md5(meta['job_url'].encode()).hexdigest()[:12],
                    title           = meta['title'],
                    company         = meta['company'],
                    industry_key    = industry_key,
                    industry_label  = industry_label,
                    seniority       = '',
                    location        = meta['location'],
                    date_posted     = meta['date_posted'] or meta['date_text'],
                    description     = description,
                    skills_tags     = '',
                    employment_type = '',
                    source_platform = 'linkedin',
                    source_url      = meta['job_url'],
                )
                h = p.content_hash()
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    postings.append(p)
                    new_this_page += 1
                random_delay(1.5, 3.5)

            log.info(f'  new unique jobs this page: {new_this_page} | total so far: {len(postings)}')

            if new_this_page == 0:
                empty_streak += 1
                if empty_streak >= EMPTY_THRESHOLD:
                    log.info('Two consecutive pages with no new jobs — moving on.')
                    break
            else:
                empty_streak = 0

            page += 1
            random_delay(5, 9)

        log.info(f'LinkedIn | {industry_key} | "{search_term}" → {len(postings)} postings')
        return postings

    # ------------------------------------------------------------------
    def close(self) -> None:
        """Close Chrome cleanly."""
        try:
            self.driver.quit()
        except Exception:
            pass


print('LinkedInScraper class defined.')


LinkedInScraper class defined.


## 4. Indeed Scraper (Fallback)

Used when LinkedIn returns insufficient results or blocks the scraper. Queries `indeed.com/jobs` via `requests` + `BeautifulSoup`.  
Indeed does not paginate historical dates reliably, so we spread queries across different role terms within each time bin.

In [26]:
class IndeedScraper:
    """
    requests + BeautifulSoup scraper for Indeed job postings.

    Indeed uses a 'fromage' (from-age-in-days) parameter for date filtering.
    For historical queries we page through sorted results and filter by date.
    """

    BASE_URL = os.getenv('INDEED_BASE_URL', 'https://www.indeed.com')

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(get_headers())

    # ------------------------------------------------------------------
    def _search_url(self, query: str, location: str = 'United States',
                    page_offset: int = 0, sort: str = 'date') -> str:
        params = {
            'q':    query,
            'l':    location,
            'sort': sort,
            'start': str(page_offset * 10),
        }
        return self.BASE_URL + '/jobs?' + urllib.parse.urlencode(params)

    # ------------------------------------------------------------------
    def _parse_date(self, raw: str) -> str:
        """
        Convert Indeed's relative date strings ('30+ days ago', 'Just posted') to
        an approximate ISO date. Returns '' if unparseable.
        """
        raw = raw.lower().strip()
        today = datetime.utcnow().date()
        if 'just' in raw or 'today' in raw:
            return str(today)
        m = re.search(r'(\d+)\+?\s*day', raw)
        if m:
            return str(today - timedelta(days=int(m.group(1))))
        return ''

    # ------------------------------------------------------------------
    def _fetch_page(self, url: str) -> Optional[BeautifulSoup]:
        """GET a page with retry logic; return BeautifulSoup or None."""
        for attempt in range(4):
            try:
                self.session.headers.update({'User-Agent': UA.random})
                resp = self.session.get(url, timeout=15)
                if resp.status_code == 200:
                    return BeautifulSoup(resp.text, 'lxml')
                if resp.status_code in (429, 403):
                    wait = 30 * (2 ** attempt)
                    log.warning(f'Indeed rate limited ({resp.status_code}) — waiting {wait}s')
                    time.sleep(wait)
            except requests.RequestException as exc:
                log.warning(f'Request error: {exc} — retry {attempt + 1}')
                time.sleep(10)
        return None

    # ------------------------------------------------------------------
    def _scrape_detail(self, job_url: str) -> Dict:
        """Fetch the full job description from the detail page."""
        result = {'description': '', 'employment_type': '', 'seniority': '', 'skills_tags': ''}
        soup = self._fetch_page(job_url)
        if not soup:
            return result

        desc_el = soup.select_one('#jobDescriptionText') or soup.select_one('div.jobsearch-jobDescriptionText')
        if desc_el:
            result['description'] = desc_el.get_text(separator=' ').strip()

        emp_el = soup.select_one('div[data-testid="job-type-display"]')
        if emp_el:
            result['employment_type'] = emp_el.get_text().strip()

        return result

    # ------------------------------------------------------------------
    def search(self, industry_key: str, query: str,
               start_date: str, end_date: str,
               max_results: int = 50) -> List[JobPosting]:
        """Collect postings for a given query, filtering by date window."""
        industry_cfg = INDUSTRIES[industry_key]
        postings: List[JobPosting] = []
        seen_hashes: set = set()
        page = 0
        start_dt = datetime.fromisoformat(start_date).date()
        end_dt   = datetime.fromisoformat(end_date).date()

        while len(postings) < max_results:
            url  = self._search_url(query, page_offset=page)
            soup = self._fetch_page(url)
            if soup is None:
                break

            cards = soup.select('div.job_seen_beacon') or soup.select('div.tapItem')
            if not cards:
                log.info(f'Indeed | {industry_key} | page {page} — no cards, stopping.')
                break

            for card in cards:
                if len(postings) >= max_results:
                    break

                title_el   = card.select_one('h2.jobTitle span') or card.select_one('h2 a span')
                company_el = card.select_one('span.companyName') or card.select_one('div[data-testid="company-name"]')
                loc_el     = card.select_one('div.companyLocation') or card.select_one('div[data-testid="text-location"]')
                date_el    = card.select_one('span.date') or card.select_one('span[data-testid="myJobsStateDate"]')
                link_el    = card.select_one('a[data-jk]') or card.select_one('a.jcs-JobTitle')

                if not title_el or not link_el:
                    continue

                raw_date   = date_el.get_text().strip() if date_el else ''
                date_posted = self._parse_date(raw_date)

                # Date window filter
                if date_posted:
                    dt = datetime.fromisoformat(date_posted).date()
                    if not (start_dt <= dt <= end_dt):
                        continue

                jk  = link_el.get('data-jk', '')
                job_url = f'{self.BASE_URL}/viewjob?jk={jk}' if jk else ''

                detail = self._scrape_detail(job_url) if job_url else {}
                if not detail.get('description'):
                    continue

                p = JobPosting(
                    job_id          = jk or hashlib.md5(job_url.encode()).hexdigest()[:12],
                    title           = title_el.get_text().strip(),
                    company         = company_el.get_text().strip() if company_el else '',
                    industry_key    = industry_key,
                    industry_label  = industry_cfg['label'],
                    seniority       = '',
                    location        = loc_el.get_text().strip() if loc_el else '',
                    date_posted     = date_posted,
                    description     = detail['description'],
                    skills_tags     = '',
                    employment_type = detail.get('employment_type', ''),
                    source_platform = 'indeed',
                    source_url      = job_url,
                )

                h = p.content_hash()
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    postings.append(p)

                random_delay(2, 5)

            page += 1
            random_delay(5, 10)

        log.info(f'Indeed | {industry_key} | "{query}" → {len(postings)} postings')
        return postings


print('IndeedScraper class defined.')

IndeedScraper class defined.


## 5. Wayback Machine Scraper (Pre-2019 Historical Data)

For the `pre_gpt3` bin (2015–2020), LinkedIn results are rarely archived usefully.  
Instead we query the CDX Index API to find archived **Indeed** search-results pages, retrieve the snapshot HTML, and parse it with the same Indeed logic.

CDX API endpoint: `http://web.archive.org/cdx/search/cdx`

In [27]:
class WaybackScraper:
    """
    Retrieve archived job-search pages from the Wayback Machine (archive.org).

    Strategy
    --------
    1. Use the CDX Search API to enumerate available snapshots of
       `indeed.com/jobs?q=<role>` within a date range.
    2. Fetch each snapshot URL and parse it as an Indeed results page.
    3. Parse individual job detail URLs embedded in the snapshot, then
       attempt to retrieve those archived detail pages too.
    """

    CDX_API = 'http://web.archive.org/cdx/search/cdx'
    WB_BASE = 'https://web.archive.org/web/{}id_/{}'   # timestamp, original_url
    RATE    = float(os.getenv('WAYBACK_RATE_LIMIT_SECONDS', 2))

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(get_headers('https://web.archive.org'))

    # ------------------------------------------------------------------
    def _cdx_snapshots(self, original_url: str,
                        from_dt: str, to_dt: str,
                        limit: int = 50) -> List[Dict]:
        """
        Query the CDX API for archived snapshots of `original_url`
        between `from_dt` and `to_dt` (format YYYYMMDD).
        Returns list of {timestamp, url} dicts.
        """
        params = {
            'url':       original_url,
            'output':    'json',
            'from':      from_dt.replace('-', ''),
            'to':        to_dt.replace('-', ''),
            'fl':        'timestamp,original',
            'filter':    'statuscode:200',
            'collapse':  'timestamp:8',  # one per day
            'limit':     str(limit),
        }
        try:
            resp = self.session.get(self.CDX_API, params=params, timeout=20)
            resp.raise_for_status()
            rows = resp.json()
            # First row is headers
            if not rows or len(rows) < 2:
                return []
            headers = rows[0]
            return [dict(zip(headers, row)) for row in rows[1:]]
        except Exception as exc:
            log.warning(f'CDX API error: {exc}')
            return []

    # ------------------------------------------------------------------
    def _fetch_snapshot(self, timestamp: str, original_url: str) -> Optional[BeautifulSoup]:
        """Retrieve an archived page and return parsed HTML."""
        wb_url = self.WB_BASE.format(timestamp, original_url)
        try:
            time.sleep(self.RATE)
            resp = self.session.get(wb_url, timeout=30)
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, 'lxml')
        except requests.RequestException as exc:
            log.warning(f'Wayback fetch error: {exc}')
        return None

    # ------------------------------------------------------------------
    def _parse_indeed_archive(self, soup: BeautifulSoup,
                               timestamp: str, industry_key: str) -> List[Dict]:
        """Extract job listings from an archived Indeed results page."""
        results = []
        # Indeed results HTML has changed over the years — try multiple selectors
        cards = (soup.select('div.result') or
                 soup.select('div.job_seen_beacon') or
                 soup.select('div.tapItem'))

        posted_date = f'{timestamp[:4]}-{timestamp[4:6]}-{timestamp[6:8]}'

        for card in cards:
            title_el   = card.select_one('a.jobtitle') or card.select_one('h2.jobTitle span')
            company_el = card.select_one('span.company') or card.select_one('span.companyName')
            loc_el     = card.select_one('span.location') or card.select_one('div.companyLocation')
            link_el    = card.select_one('a[href*="/viewjob"]') or card.select_one('a[data-jk]')

            if not title_el:
                continue

            href = link_el['href'] if link_el and link_el.has_attr('href') else ''
            if href.startswith('/'):
                href = 'https://www.indeed.com' + href

            # Attempt to retrieve archived detail page
            desc = ''
            if href:
                detail_soup = self._fetch_snapshot(timestamp, href)
                if detail_soup:
                    desc_el = (detail_soup.select_one('#jobDescriptionText') or
                               detail_soup.select_one('div.jobsearch-jobDescriptionText'))
                    if desc_el:
                        desc = desc_el.get_text(separator=' ').strip()

            results.append({
                'title':    title_el.get_text().strip(),
                'company':  company_el.get_text().strip() if company_el else '',
                'location': loc_el.get_text().strip() if loc_el else '',
                'date_posted': posted_date,
                'description': desc,
                'source_url': href,
            })
        return results

    # ------------------------------------------------------------------
    def search(self, industry_key: str, role: str,
               start_date: str, end_date: str,
               max_results: int = 50) -> List[JobPosting]:
        """Scrape archived Indeed pages for a given role and date range."""
        industry_cfg = INDUSTRIES[industry_key]
        postings: List[JobPosting] = []
        seen_hashes: set = set()

        original_url = f'https://www.indeed.com/jobs?q={urllib.parse.quote(role)}&sort=date'
        snapshots = self._cdx_snapshots(original_url, start_date, end_date, limit=100)
        random.shuffle(snapshots)  # randomise to spread across the date range

        log.info(f'Wayback | {industry_key} | "{role}" → {len(snapshots)} snapshots found')

        for snap in snapshots:
            if len(postings) >= max_results:
                break
            soup = self._fetch_snapshot(snap['timestamp'], snap['original'])
            if not soup:
                continue

            raw_list = self._parse_indeed_archive(soup, snap['timestamp'], industry_key)

            for item in raw_list:
                if len(postings) >= max_results:
                    break
                if not item.get('description') or len(item['description'].split()) < 30:
                    continue

                p = JobPosting(
                    job_id          = hashlib.md5(item['source_url'].encode()).hexdigest()[:12],
                    title           = item['title'],
                    company         = item['company'],
                    industry_key    = industry_key,
                    industry_label  = industry_cfg['label'],
                    seniority       = '',
                    location        = item['location'],
                    date_posted     = item['date_posted'],
                    description     = item['description'],
                    skills_tags     = '',
                    employment_type = '',
                    source_platform = 'wayback',
                    source_url      = item['source_url'],
                )
                h = p.content_hash()
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    postings.append(p)

        log.info(f'Wayback | {industry_key} | "{role}" → {len(postings)} postings collected')
        return postings


print('WaybackScraper class defined.')

WaybackScraper class defined.


## 6. Temporal Sampling Strategy

In [28]:
def build_scraping_plan() -> List[Dict]:
    """
    Build a flat list of scraping tasks.

    Three task types, each routed to a different scraper:
      - kind="linkedin" : (industry, search_term) — runs once per term, fills
                          recent bins (gpt4_harvey, gpt4o_present) from live search.
      - kind="indeed"   : (industry, bin)         — boolean Indeed query, date-filtered.
      - kind="wayback"  : (industry, bin)         — archive.org snapshots, pre-2019.

    Bins are bound to specific scrapers via LINKEDIN_BINS / WAYBACK_BINS in cell 4.
    """
    tasks: List[Dict] = []

    # ----- LinkedIn: one task per (industry, search_term) -----
    li_target = TARGET_PER_BIN * len(LINKEDIN_BINS)   # total across recent bins
    for ind_key, ind_cfg in INDUSTRIES.items():
        terms = ind_cfg.get('linkedin_search_terms', [])
        if not terms:
            continue
        per_term = max(10, li_target // len(terms))
        for term in terms:
            tasks.append({
                'kind':         'linkedin',
                'industry_key': ind_key,
                'search_term':  term,
                'start_date':   min(s for b, s, e in TIME_BINS if b in LINKEDIN_BINS),
                'end_date':     max(e for b, s, e in TIME_BINS if b in LINKEDIN_BINS),
                'max_results':  per_term,
            })

    # ----- Indeed + Wayback: one task per (industry, bin) -----
    for ind_key in INDUSTRIES:
        for bin_label, bin_start, bin_end in TIME_BINS:
            if bin_label in WAYBACK_BINS:
                tasks.append({
                    'kind':         'wayback',
                    'industry_key': ind_key,
                    'bin_label':    bin_label,
                    'start_date':   bin_start,
                    'end_date':     bin_end,
                    'max_results':  TARGET_PER_BIN,
                })
            else:
                # Indeed covers everything else (including recent bins as backup)
                tasks.append({
                    'kind':         'indeed',
                    'industry_key': ind_key,
                    'bin_label':    bin_label,
                    'start_date':   bin_start,
                    'end_date':     bin_end,
                    'max_results':  TARGET_PER_BIN,
                })

    random.shuffle(tasks)
    return tasks


PLAN = build_scraping_plan()
print(f'Scraping plan: {len(PLAN)} tasks')
print(f'  linkedin tasks: {sum(1 for t in PLAN if t["kind"] == "linkedin")}')
print(f'  indeed tasks:   {sum(1 for t in PLAN if t["kind"] == "indeed")}')
print(f'  wayback tasks:  {sum(1 for t in PLAN if t["kind"] == "wayback")}')
import pprint
pprint.pprint(PLAN[:3])


Scraping plan: 95 tasks
  linkedin tasks: 70
  indeed tasks:   20
  wayback tasks:  5
[{'bin_label': 'pre_gpt3',
  'end_date': '2020-09-30',
  'industry_key': 'pharma_chem',
  'kind': 'wayback',
  'max_results': 200,
  'start_date': '2015-01-01'},
 {'end_date': '2026-05-11',
  'industry_key': 'patent_ip',
  'kind': 'linkedin',
  'max_results': 30,
  'search_term': 'IP',
  'start_date': '2023-03-01'},
 {'end_date': '2026-05-11',
  'industry_key': 'patent_ip',
  'kind': 'linkedin',
  'max_results': 30,
  'search_term': 'innovation',
  'start_date': '2023-03-01'}]


## 7. Main Execution Loop

Iterates through the plan, dispatches to the appropriate scraper, writes results incrementally (so progress is not lost on crash), and logs bin-level counts.

In [29]:
def run_scraping(plan: List[Dict], headless: bool = False) -> Dict:
    """
    Execute the full scraping plan, dispatching each task by its `kind`.

    Behaviour
    ---------
    - LinkedIn is created lazily on the first linkedin task and reused.
      If LinkedIn login fails, all linkedin tasks are skipped (Indeed picks up slack).
    - Indeed and Wayback are stateless and constructed up front.
    - Results are persisted incrementally to `data/raw/postings_<industry>.csv`
      so progress survives crashes.
    - Returns count_tracker = {industry_key: {bin_label: int}}.
    """
    li_scraper:   Optional[LinkedInScraper] = None
    li_available  = None    # tri-state: None=untried, True/False=tried

    indeed_scraper  = IndeedScraper()
    wayback_scraper = WaybackScraper()

    count_tracker: Dict[str, Dict[str, int]] = {
        k: {b[0]: 0 for b in TIME_BINS} for k in INDUSTRIES
    }

    def _ensure_linkedin() -> bool:
        """Lazy-init LinkedIn scraper on first need.

        On *renderer* timeouts (Chrome's JS process froze) we mark the scraper
        unhealthy and let the next task retry by re-initializing — this prevents
        a single hiccup during login from killing the whole LinkedIn pipeline.
        """
        nonlocal li_scraper, li_available
        if li_available is True:
            return True
        # Allow re-attempts: only treat as permanently failed after 2 init crashes
        for attempt in range(2):
            try:
                li_scraper   = LinkedInScraper(headless=headless)
                li_available = li_scraper.login()
                if li_available:
                    return True
            except Exception as exc:
                log.warning(
                    f'LinkedIn init attempt {attempt+1}/2 failed '
                    f'({exc.__class__.__name__}: {str(exc)[:120]})'
                )
                try:
                    if li_scraper is not None:
                        li_scraper.close()
                except Exception:
                    pass
                li_scraper = None
        li_available = False
        log.error('LinkedIn unavailable after retries — falling back to Indeed only.')
        return False

    try:
        for task in tqdm(plan, desc='Scraping tasks'):
            kind     = task['kind']
            ind_key  = task['industry_key']
            start    = task['start_date']
            end      = task['end_date']
            max_res  = task['max_results']
            postings: List[JobPosting] = []

            if kind == 'linkedin':
                if not _ensure_linkedin():
                    continue
                postings = li_scraper.search(
                    industry_key = ind_key,
                    search_term  = task['search_term'],
                    start_date   = start,
                    end_date     = end,
                    max_results  = max_res,
                )

            elif kind == 'indeed':
                query    = INDUSTRIES[ind_key]['indeed_query']
                postings = indeed_scraper.search(ind_key, query, start, end, max_res)

            elif kind == 'wayback':
                # Wayback uses the indeed_query as its role string
                role     = INDUSTRIES[ind_key]['indeed_query']
                postings = wayback_scraper.search(ind_key, role, start, end, max_res)

            # Persist and update counters
            out_path = DATA_DIR / f'postings_{ind_key}.csv'
            write_postings(postings, out_path)

            for p in postings:
                bin_label = assign_time_bin(p.date_posted)
                if bin_label in count_tracker[ind_key]:
                    count_tracker[ind_key][bin_label] += 1

            log.info(
                f'[{ind_key}][{kind}] {task.get("search_term") or task.get("bin_label", "")} '
                f'→ {len(postings)} postings'
            )
            random_delay(2, 5)

    finally:
        if li_scraper is not None:
            li_scraper.close()

    return count_tracker


# ── Uncomment when ready to scrape ───────────────────────────────────────────
count_tracker = run_scraping(PLAN, headless=False)
print('run_scraping() defined — uncomment the call above when ready to scrape.')


Scraping tasks:   0%|          | 0/95 [00:00<?, ?it/s]

2026-05-17 01:21:03,101 [WARNING] CDX API error: 503 Server Error: Service Unavailable for url: http://web.archive.org/cdx/search/cdx?url=https%3A%2F%2Fwww.indeed.com%2Fjobs%3Fq%3D%2528scientist%2520OR%2520chemist%2520OR%2520analyst%2520OR%2520engineer%2520OR%2520researcher%2529%2520AND%2520%2528pharmaceutical%2520OR%2520chemical%2520OR%2520biotech%2520OR%2520laboratory%2520OR%2520drug%2529%26sort%3Ddate&output=json&from=20150101&to=20200930&fl=timestamp%2Coriginal&filter=statuscode%3A200&collapse=timestamp%3A8&limit=100
2026-05-17 01:21:03,104 [INFO] Wayback | pharma_chem | "(scientist OR chemist OR analyst OR engineer OR researcher) AND (pharmaceutical OR chemical OR biotech OR laboratory OR drug)" → 0 snapshots found
2026-05-17 01:21:03,105 [INFO] Wayback | pharma_chem | "(scientist OR chemist OR analyst OR engineer OR researcher) AND (pharmaceutical OR chemical OR biotech OR laboratory OR drug)" → 0 postings collected
2026-05-17 01:21:03,106 [INFO] [pharma_chem][wayback] pre_gpt3 →

run_scraping() defined — uncomment the call above when ready to scrape.


## 8. Metadata Log

In [31]:
def generate_metadata_log(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Read all per-industry CSVs and produce a metadata summary log."""
    records = []

    for ind_key, ind_cfg in INDUSTRIES.items():
        path = data_dir / f'postings_{ind_key}.csv'
        if not path.exists():
            records.append({
                'industry_key':   ind_key,
                'industry_label': ind_cfg['label'],
                'n_postings':     0,
                'date_min':       None,
                'date_max':       None,
                'n_bins_covered': 0,
                'missing_bins':   ','.join(b[0] for b in TIME_BINS),
                'sources':        '',
                'notes':          'FILE NOT FOUND',
            })
            continue

        df = pd.read_csv(path, parse_dates=['date_posted'], dtype=str)
        df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

        # Bin coverage
        df['time_bin'] = df['date_posted'].dt.strftime('%Y-%m-%d').apply(
            lambda d: assign_time_bin(d) if pd.notna(d) else 'unknown'
        )
        covered_bins = set(df['time_bin'].unique()) - {'unknown'}
        all_bins     = {b[0] for b in TIME_BINS}
        missing_bins = all_bins - covered_bins

        # Gap analysis: flag bins with fewer than 50 postings
        bin_counts  = df['time_bin'].value_counts().to_dict()
        sparse_bins = [b for b in all_bins if bin_counts.get(b, 0) < 50]

        records.append({
            'industry_key':   ind_key,
            'industry_label': ind_cfg['label'],
            'n_postings':     len(df),
            'date_min':       df['date_posted'].min().strftime('%Y-%m-%d') if df['date_posted'].notna().any() else '',
            'date_max':       df['date_posted'].max().strftime('%Y-%m-%d') if df['date_posted'].notna().any() else '',
            'n_bins_covered': len(covered_bins),
            'missing_bins':   ','.join(sorted(missing_bins)),
            'sparse_bins':    ','.join(sorted(sparse_bins)),
            'sources':        ','.join(sorted(df['source_platform'].dropna().unique())),
            'notes':          'OK' if len(df) >= MIN_PER_INDUSTRY else f'BELOW MINIMUM ({MIN_PER_INDUSTRY})',
        })

    meta_df = pd.DataFrame(records)
    meta_df.to_csv(data_dir / 'metadata_log.csv', index=False)
    print(f'Metadata log saved to {data_dir / "metadata_log.csv"}')
    return meta_df


# ── Run after scraping ───────────────────────────────────────────────────────
meta_df = generate_metadata_log()
display(meta_df)
print('generate_metadata_log() defined — call after run_scraping().')

Metadata log saved to data/raw/metadata_log.csv


,industry_key,industry_label,n_postings,date_min,date_max,n_bins_covered,missing_bins,sparse_bins,sources,notes
0,pharma_chem,Pharmaceutical / Chemical Manufacturing,806,2026-05-11,2026-05-18,1,"copilot_chatgpt,gpt3_era,gpt4_harvey,pre_gpt3","copilot_chatgpt,gpt3_era,gpt4_harvey,gpt4o_pre...",linkedin,OK
1,legal_services,Legal Services,924,2026-05-08,2026-05-18,1,"copilot_chatgpt,gpt3_era,gpt4_harvey,pre_gpt3","copilot_chatgpt,gpt3_era,gpt4_harvey,gpt4o_pre...",linkedin,OK
2,farming_forestry,"Farming, Ranching and Forestry",1070,2026-04-11,2026-05-18,1,"copilot_chatgpt,gpt3_era,gpt4_harvey,pre_gpt3","copilot_chatgpt,gpt3_era,gpt4_harvey,gpt4o_pre...",linkedin,OK
3,insurance,Insurance,1170,2026-05-07,2026-05-18,1,"copilot_chatgpt,gpt3_era,gpt4_harvey,pre_gpt3","copilot_chatgpt,gpt3_era,gpt4_harvey,gpt4o_pre...",linkedin,OK
4,patent_ip,Patent Analysis and IP Research,1273,2026-04-30,2026-05-18,1,"copilot_chatgpt,gpt3_era,gpt4_harvey,pre_gpt3","copilot_chatgpt,gpt3_era,gpt4_harvey,gpt4o_pre...",linkedin,OK


generate_metadata_log() defined — call after run_scraping().


## 9. Quick Validation (Post-Scrape)

Run this cell after scraping to verify data integrity before moving to Notebook 2.

In [33]:
def quick_validation(data_dir: Path = DATA_DIR) -> None:
    """Print a quick summary of all scraped CSVs to catch obvious problems."""
    header = "Industry".ljust(40) + "N".rjust(6) + "Date min".rjust(12) + "Date max".rjust(12) + "  Sources"
    print(header)
    print('-' * 85)

    for ind_key, ind_cfg in INDUSTRIES.items():
        path = data_dir / f'postings_{ind_key}.csv'
        if not path.exists():
            print(ind_cfg["label"].ljust(40) + "MISSING".rjust(6))
            continue
        df = pd.read_csv(path, dtype=str)
        dates = pd.to_datetime(df['date_posted'], errors='coerce')
        sources = df['source_platform'].value_counts().to_dict()
        src_str = ' | '.join(f'{k}:{v}' for k, v in sources.items())
        dmin = dates.min().strftime('%Y-%m') if dates.notna().any() else 'N/A'
        dmax = dates.max().strftime('%Y-%m') if dates.notna().any() else 'N/A'
        print(ind_cfg["label"].ljust(40) + str(len(df)).rjust(6) + dmin.rjust(12) + dmax.rjust(12) + '  ' + src_str)


# Run after scraping
quick_validation()
print('quick_validation() defined.')

Industry                                     N    Date min    Date max  Sources
-------------------------------------------------------------------------------------
Pharmaceutical / Chemical Manufacturing    806     2026-05     2026-05  linkedin:806
Legal Services                             924     2026-05     2026-05  linkedin:924
Farming, Ranching and Forestry            1070     2026-04     2026-05  linkedin:1070
Insurance                                 1170     2026-05     2026-05  linkedin:1170
Patent Analysis and IP Research           1273     2026-04     2026-05  linkedin:1273
quick_validation() defined.
